# KarstGeophysics Project
First look at shot gathers from field data

In [ ]:
from pathlib import Path
from datetime import date
import sys
import obspy

# Parent directory containing segy_tools/ and specfem_tools/.
repo_dir = Path("~/Developer/karstgeo/01_modeling/specfem2d").expanduser()
if not (repo_dir / "segy_tools").exists():
    raise FileNotFoundError(
        f"Could not find segy_tools/ under repo_dir={repo_dir}. "
        "Set repo_dir to the parent directory of segy_tools/."
    )

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
    
import numpy as np
import pandas as pd
import obspy
from obspy import Stream, Trace, UTCDateTime
from obspy.core import AttribDict

from specfem_tools.io import read_sem_gather
from segy_tools.gather import plot_wiggle_gather_from_stream
from segy_tools.io import read_su_file, write_segy
from segy_tools.headers import make_trace_header
from segy_tools.plotting import plot_wiggle_gather, plot_difference_gathers


In [ ]:

DOWNLOAD_DIR = Path('~/Downloads').expanduser()
geode_folders = sorted([f for f in DOWNLOAD_DIR.iterdir() if f.is_dir() and f.name.startswith('LBS_')])

feet_to_meters = 0.3048
scale=1.0
normalize=True


for daynum, data_dir in enumerate(geode_folders):
    print(f"Found data directory: {data_dir}")
    mdy = data_dir.name.split('_')[1]  # Extract month, day, year
    print(f"Extracted date components: {mdy}")
    day = date(2000+int(mdy[4:6]), int(mdy
                                       [0:2]), int(mdy[2:4]))
    print(f"Data for date: {day}")
    print(f"Processing data in directory: {data_dir}")
    alldatfiles = sorted(list(data_dir.glob('*.dat')))
    
    outdir = data_dir / "plots"
    outdir.mkdir(exist_ok=True)
    
    segydir = data_dir / "segy"
    segydir.mkdir(exist_ok=True)    
    
    for datfilenum, datfile in enumerate(alldatfiles):
        print(f"Processing file: {datfile}")
        try:
            st = obspy.read(str(datfile), format='SEG2')
            N = len(st)
            print(f"Read {N} traces from file: {datfile}")
            for i, tr in enumerate(st):
                tr.stats.station = f"CHA{i+1:02d}"  # Use CHA01, CHA02, etc. for station names
                if N==24: # really need a configuration file in each folder to specify these things, but for now we can just hard-code
                    tr.stats.network = "GV"
                    receiver_spacing_m = 5 * feet_to_meters  # or is it 5 feet,which is slightly different? receiver spacing for GV
                    source_x_m = 0  # Source at x=0 for GV
                    first_receiver_x_m = 30 * feet_to_meters  # First receiver at x=1.5 m for GV
                elif N==72:
                    tr.stats.network = "LB"
                    receiver_spacing_m = 1  # 1-m forced for now for receiver spacing for LB. but we also did some with 2-m spacing
                    source_x_m = 0  # Source at x=0 for LB
                    first_receiver_x_m = 1  # First receiver at x=1 m for LB
                else:
                    tr.stats.network = "XX"  # Use XX for unknown network
                if tr.stats.sampling_rate < 1000:
                    tr.stats.channel = f"DHZ"  # Use HHZ for vertical component
                else:
                    tr.stats.channel = f"GHZ"  # Use EHZ for high-sampling-rate vertical component
                if tr.stats.network == "GV":
                    tr.stats.location = f"{datfilenum:02d}"  # Use file number as location code
                else:
                    tr.stats.location = f"{daynum:02d}"  # Use day number as location code
            print(st.__str__(extended=True ))
            outfile = outdir / f"{datfile.stem}_shot_gather.png"
            segyfile = segydir / f"{datfile.stem}.segy"
            plot_wiggle_gather_from_stream(
                st,
                fallback_receiver_spacing_m=receiver_spacing_m,
                fallback_first_receiver_x_m=first_receiver_x_m,
                fallback_source_x_m=source_x_m,
                tmin=0.0,
                tmax=st[0].stats.npts / st[0].stats.sampling_rate,
                scale=scale,
                normalize=normalize,
                title=f"Shot gather for file {datfile.name} in directory {data_dir.name}",
                outfile=outfile,
            )
            #write_segy(st, segyfile)
            st.write(str(segyfile), format='SEGY', byteorder='>')
            #input("Press Enter to continue to the next file...")
        except Exception as e:
            print(f"Error reading file {datfile} as SEG2 format: {e}")
        

In [ ]:

DOWNLOAD_DIR = Path('~/Downloads').expanduser()
geode_folders = sorted([f for f in DOWNLOAD_DIR.iterdir() if f.is_dir() and f.name.startswith('LBSSP_')])

feet_to_meters = 0.3048
scale=1.0
normalize=True

freq = [200, 400]
apply_bandpass_filter = True


for daynum, data_dir in enumerate(geode_folders):
    print(f"Found data directory: {data_dir}")
    mdy = data_dir.name.split('_')[1]  # Extract month, day, year
    print(f"Extracted date components: {mdy}")
    day = date(2000+int(mdy[4:6]), int(mdy
                                       [0:2]), int(mdy[2:4]))
    print(f"Data for date: {day}")
    print(f"Processing data in directory: {data_dir}")
    alldatfiles = sorted(list(data_dir.glob('*.dat')))
    
    outdir = data_dir / "plots"
    outdir.mkdir(exist_ok=True)
    
    segydir = data_dir / "segy"
    segydir.mkdir(exist_ok=True)    
    
    for datfilenum, datfile in enumerate(alldatfiles):
        print(f"Processing file: {datfile}")
        try:
            st = obspy.read(str(datfile), format='SEG2')
            N = len(st)
            print(f"Read {N} traces from file: {datfile}")
            for i, tr in enumerate(st):
                tr.stats.station = f"CHA{i+1:02d}"  # Use CHA01, CHA02, etc. for station names
                if N==24: # really need a configuration file in each folder to specify these things, but for now we can just hard-code
                    tr.stats.network = "GV"
                    receiver_spacing_m = 5 * feet_to_meters  # or is it 5 feet,which is slightly different? receiver spacing for GV
                    source_x_m = 0  # Source at x=0 for GV
                    first_receiver_x_m = 30 * feet_to_meters  # First receiver at x=1.5 m for GV
                elif N==72:
                    tr.stats.network = "LB"
                    duration = tr.stats.npts / tr.stats.sampling_rate
                    if duration < 0.5:
                        receiver_spacing_m = 1  # 1-m forced for now for receiver spacing for LB
                        first_receiver_x_m = 87
                        source_pos = datfilenum
                        first_shot_offset_m = -4.5  # First shot is 4.5 m before first receiver for LB
                    else:
                        receiver_spacing_m = 2  # 2-m spacing for longer duration LB gathers
                        first_receiver_x_m = 52
                        source_pos = datfilenum - 47
                        first_shot_offset_m = -4  # First shot is 4 m before first receiver for longer duration LB gathers
                    source_x_m = first_receiver_x_m + source_pos * receiver_spacing_m * 2 + first_shot_offset_m

                else:
                    tr.stats.network = "XX"  # Use XX for unknown network
                if tr.stats.sampling_rate < 1000:
                    tr.stats.channel = f"DHZ"  # Use HHZ for vertical component
                else:
                    tr.stats.channel = f"GHZ"  # Use EHZ for high-sampling-rate vertical component
                if tr.stats.network == "GV":
                    tr.stats.location = f"{datfilenum:02d}"  # Use file number as location code
                else:
                    tr.stats.location = f"{daynum:02d}"  # Use day number as location code
            print(st.__str__(extended=True ))
            if apply_bandpass_filter:
                outfile = outdir / f"{datfile.stem}_shot_gather_filtered_300Hz.png"
                st.detrend(type='demean')
                st.filter('bandpass', freqmin=freq[0], freqmax=freq[1], corners=4, zerophase=True)
            else:
                outfile = outdir / f"{datfile.stem}_shot_gather.png"
            segyfile = segydir / f"{datfile.stem}.segy"
            
            if outfile.exists():
                print(f"Output file {outfile} already exists. Skipping plotting for this file.")
            else:
                plot_wiggle_gather_from_stream(
                    st,
                    fallback_receiver_spacing_m=receiver_spacing_m,
                    fallback_first_receiver_x_m=first_receiver_x_m,
                    fallback_source_x_m=source_x_m,
                    tmin=0.0,
                    tmax=duration,
                    scale=scale,
                    normalize=normalize,
                    title=f"Shot {source_pos+1} at x={source_x_m:.1f} m for {data_dir.name}/{datfile.name}",
                    outfile=outfile,
                )
            
            if not segyfile.exists():
                #write_segy(st, segyfile)
                st.write(str(segyfile), format='SEGY', byteorder='>')
            #input("Press Enter to continue to the next file...")
        except Exception as e:
            print(f"Error reading file {datfile} as SEG2 format: {e}")
        